# Summary of most used segmentation metrics

- Metrics help identify algorithms that perform well interns of desired criteria
- Also, parameter tuning can be chosen based on evaluation metrics

In [49]:
import metrics as mtr
import numpy as np
from scipy.ndimage import label
import torch
from torchmetrics.classification import BinaryHammingDistance
from PIL import Image
import torchvision.transforms as T

### Importing masks (Is the same Mask)

In [ ]:
transform=T.ToTensor()

# GROUND TRUTH MASK 
truth_path="/Users/jorgeandresjolahernandez/Desktop/CV/SegmentationStudy/utils/eval/truth.png"
truth_img=Image.open(truth_path).convert("L")
# PREDICTED MASK
pred_path="/Users/jorgeandresjolahernandez/Desktop/CV/SegmentationStudy/utils/eval/pred.png"
pred_img=Image.open(pred_path).convert("L")



truth_tensor=transform(truth_img)
pred_tensor=transform(pred_img)

truth_tensor = (truth_tensor > 0.5).to(torch.int32)
pred_tensor = (pred_tensor > 0.5).to(torch.int32)

# The module works with numpy arrays instead of tensors
truth = truth_tensor.squeeze().numpy()
pred = pred_tensor.squeeze().numpy()


print(truth.shape,'\n',pred.shape)



(512, 512) 
 (512, 512)


In [53]:
# Start the evaluation using the Metrics Class
test=mtr.Metrics(truth,pred)

## 1. Confusion Matrix Components

It visualizes the performance of the model, showing the count of correct and incorrect prediction for each class.

$$

\text{Confusion Matrix} =
\begin{bmatrix}
\text{TP} & \text{FP} \\
\text{FN} & \text{TN}
\end{bmatrix}
$$

In [29]:
test.calc_ConfusionMatrix()

(np.int64(6152), np.int64(255992), np.int64(0), np.int64(0))

### 1.1 True Positive

Pixels correctly predicted as the target class

In [30]:
test.calc_TruePositive()

np.int64(6152)

### 1.2 True Negative

Pixels correctly predicted as background

In [31]:
test.calc_TrueNegative()

np.int64(255992)

### 1.3 False Positives

Background pixels incorrectly predicted as the class

In [32]:
test.calc_FalsePositive()

np.int64(0)

### 1.4 False Negatives

Class pixels incorrectly predicted as background.

In [33]:
test.calc_FalseNegative()

np.int64(0)

## 2. Accuracy

Overall percentage of correctly classified pixels

**Pros:** Simple and intuitive. <br>
**Cons:** Misleading for imbalanced data. <br>
**Use case:** Only meaningful when class distribution is balanced. <br>
**Possible range:** 0 --> 1 <br>
**Ideal Value:** 1 <br>

$$
\text{Accuracy}: \frac{TP+TN}{TP+TN+FP+FN}
$$

In [34]:
test.calc_Accuracy_CM()

np.float64(1.0)

## 3. Precision

Of all pixels predicted as belonging to the target class (foreground), how many actually belong to that class in the ground truth.

**Pros:** Penalizes false positives. <br>
**Cons:** Doesn’t indicate whether the model missed true foreground pixels <br>
**Use case:** Important when false detection are costly. <br>
**Possible range:** 0 --> 1 <br>
**Ideal Value:** 1 <br>

$$
\text{Precision}: \frac{TP}{TP+FP}
$$

In [35]:
test.calc_Precision_CM()

np.float64(1.0)

## 4 . Recall (Sensitivity)

Of all pixels truly belonging to the target class (foreground), how many are correctly predicted by the model. <br>

**Pros:** Penalizes false negatives (missed detections). <br>
**Cons:** Can still be high if the model over-predicts the class <br>
**Use case:** Important when missing the class is more costly <br>
**Possible range:** 0 --> 1 <br>
**Ideal Value:** 1 <br>


$$
\text{Sensitivity or Recall}:\frac{TP}{TP+FN}
$$

In [36]:
test.calc_Sensitivity_CM()

np.float64(1.0)

## 5. Specificity

Of all pixels truly belonging to the background (negative class), how many are correctly predicted as background by the model. <br>

**Pros:** Measures how well the model avoids false positives (background pixels incorrectly labeled as foreground). <br>
**Cons:** Does not indicate whether the model correctly detects the foreground (missed positives). <br>
**Use case:** Important when background dominates the image and false detections are costly <br>
**Possible range:** 0 --> 1 <br>
**Ideal Value:** 1 <br>

$$
\text{Specificity}: \frac{TN}{TN+FP}
$$

In [37]:
test.calc_Specificity_CM()

np.float64(1.0)

## 6. F1 Score

Harmonic mean of Precision and Recall. <br>

**Pros:** Provides a single metric balancing false positives and false negatives. Allows a far assesment balance <br>
**Cons:** Doesn’t differentiate whether errors are due to overprediction or missed detections. <br>
**Use case:** Commonly used to report overall segmentation quality. <br>
**Possible range:** 0 --> 1 <br>
**Ideal Value:** 1 <br>

$$
F_1 = 2 * \frac{\text{Precision}*\text{Recall}}{\text{Precision}+\text{Recall}}
$$

In [38]:
test.calc_F1_score()

np.float64(1.0)

## 7. Fβ​ Score

Weighted harmonic mean of Precision and Recall, controlled by parameter β (Penalty). <br>

**Pros:** Allows you to prioritize Recall or Precision depending on application needs. <br>
**Cons:** The choice of β is subjective; must be justified for your use case. <br>
**Use case:** Useful in scenarios where missing positive pixels (β > 1) or false alarms (β < 1) have different costs, e.g., medical lesion detection, deforestation mapping. <br>
**Possible range:** 0 --> 1 <br>
**Ideal Value:** 1 <br>

$$
F_\beta = (1+\beta^2)\frac{\text{Precision}*\text{Recall}}{(\beta^2*\text{Precision})+\text{Recall}} \\ . \\ \beta > 1 : \text{emphasizes Recall (penalizes false negatives more)} \\ \beta < 1 : \text{emphasizes Precision (penalizes false positives more)}\\\beta = 1 : \text{equivalent to F1 Score (balanced)}
$$

In [54]:
test.calc_F2_score(1)

np.float64(1.0)

## 8. Dice Coefficient

Measures the overlap between the predicted foreground and true foreground regions. <br>
Symmetrical : doesn't matter if we consider the ground truth image as the reference set or the predicted images<br>

**Pros:** Penalizes both false positives and false negatives, but gives more weight to true positives. <br>
**Cons:** Sensitive to class imbalance (small foreground regions can lead to lower Dice)<br>
**Use case:** When accurate detection of small foreground regions is critical <br>
**Possible range:** 0 --> 1 <br>
**Ideal Value:** 1 <br>

$$
DSC= \frac{2TP}{2TP + FP + FN}
$$

In [40]:
test.calc_DSC_CM()

np.float64(1.0)

## 9. Intersection over Union (IoU / Jaccard Index)

Compares ground truth segmentation with predicted segmentation giving a degree of overlap betwen the two regions <br>
Jaccard Index and Dice coefficient are equivalent in applications with high class imbalances, such as medical image segmentation, where the positive class outnumbered by the negative class<br>

**Pros:** It enhances other measures, such as the F-measure <br>
**Cons:**  It might ignore local distortions, is a global metric. Could be affected by the image class balance, just like any other ratio-based statistic<br>
**Use case:** Useful when evaluating overall pixel-level agreement. <br>
**Possible range:** 0 --> 1 <br>
**Ideal Value:** 1 <br>

$$
IoU : \frac{TP}{TP+FP+FN}
$$

In [41]:
test.calc_IoU_CM()

np.float64(1.0)

## 10. Tversky Index (TI)

DSC equally balance FP and FN, so, in order to address this class imbalance, it introduces two new weighting parameters alpha, beta<br>

**Pros:** Allows prioritizing Precision or Recall depending on application needs. More flexible than Dice when class imbalance exists.<br>
**Cons:**  Requires choosing α and β, which can be subjective. Interpretation depends on the chosen weights.<br>
**Use case:** Useful in imbalanced segmentation problems where false negatives are more critical than false positives, <br>
**Possible range:** 0 --> 1 <br>
**Ideal Value:** 1 <br>

$$
TI = \frac{TP}{TP+\alpha FP+ \beta FN} \\ . \\ \text{If} \ \alpha , \beta = 0.5 --> TI=DSC \\ \text{If} \ \alpha = 1 \ \text{and} \ \beta = 0 --> TI=\text{Precision} \\ \text{If} \ \alpha = 0 \ \text{and} \ \beta = 1 --> TI=\text{Recall}
$$

In the context of class imbalances, DSC or IoU will not work well since they will favor the mayority class

In [42]:
test.calc_Tversky_CM(alpha=0.5,beta=0.5)

np.float64(1.0)

## 11. AUC

Metric or degree of separability

- The greater AUC the better the model predict certain classes

In [43]:
test.calc_AUC_trapezoid()

np.float64(1.0)

## 12. Matthews Correlation Coefficient (MCC)

Based on the parts of Confusion Matrix

$$
MCC=\frac{TP*TN-FP*FN}{\sqrt{(TP+FP)(TP+FN)(TN+FP)(TN+FN)}}
$$

From -1 to 1: 
- 1 Represents ideal forecast
- 0 Represents no better than a haphazard estimate
- -1 Represents complete discrepancy

It considers all possible outcomes of the CM - True and false, positivie and negative.

To get high MCC, the classifier must have high TP and TN, low FP and FN.

Considred by many to be a good statistical accuracy measure when dataset is unbalanced.


In [44]:
test.calc_MCC()

np.float64(1.0)

## 13 Hamming Distance

Correspondences between regions in two segmentations

$$
D_H(S, R) = \sum_{r_i \in R} \sum_{\substack{s_k \neq s_j \\ s_k \cap r_i \neq \emptyset}} |r_i \cap s_k|
$$


S -> Segmentation result

R --> Reference segmentation

$ \text{For every region } r_i \text{ in R, we look at which regions from S overlap with it}$
$\text{If a region in R overlap several regions in S }\\ \text{If a region R overlap several regions in S --> DH (S,R) is larger --> More dissimilarity}$

They do a bidirectional comparison so they do the same for DH(R,S) --> This is because the relationship is not necessarily symmetric

After this, they normalized region-based evaluation metrics

$$
p=1-\frac{D_H(SR)+D_H(RS)}{2 x \left|S\right|}
$$

$|S|:$ is the total number of pixels

$p$ goes from 0 to 1

Higher values of $p$ represent segmentation are identical

In [45]:
test.calc_Hamm()

tensor(0.)

## 14 Binary Average Precision

$$
AP=\frac{1}{n}\sum_{i=1}P(r_i)
$$

Popular metric used to summarize a Precision-Recall curve in information retrieval and object detection.

In [46]:
test.calc_AverPres()

⚠️ Prediction array must contain probabilities (floats between 0 and 1).


## 15 Local Consistency Error (LCE)

Segmentation Structure or granularity

Quantify consistency Image segmentation with different levels of granularity. LCE enables the refinmenet of labeling between the segmentation and the ground truth.

S and R might differ in how finely they divide the image

$$
LCE(S,R)=\frac{1}{N}\sum_i min(E(S,R,p_i),E(R,S,p_i))
$$

$$N: \text{Total of pixels} \\
p_i: \text{Each individual pixel}\\
E(S,R,p_i): \text{Function that measure how consistent the labeling of pixel} p_i \text{is between S and R} \\
min(): \text{Is to get the smallest (best) error between two directions}$$

Basically, How inconsistent is S with R around pixel i.

It's local because the consistency is measured pixel by pixel, focusing on local neighborhoods rather than the image as a whole. --> Even small boundary mismatches will increase the error.

1 --> Perfect match between segmentations

0 --> Completely inconsistent segmentations

In [47]:
test.calc_LCE()

KeyboardInterrupt: 

## 16 Global Consistency Error

Looks at the overall (global) inconsistency between S and R, not just at individual pixels

$$
GCE(S,R)=\frac{1}{N} min(\sum_{p_i}E(S,R,p_i), \sum_{p_i}E(R,S,p_i))
$$

In [48]:
test.calc_GCE()

KeyboardInterrupt: 